In [1]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import xlsxwriter
from itertools import combinations
import os



/tmp/ipykernel_736188/154882641.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:


def get_folder_paths(folder_path):
    """
    Gets all the paths of folders within a specific folder, excluding .ipynb_checkpoints.

    Args:
        folder_path (str): The path to the folder.

    Returns:
        list: A list of all folder paths within the specified folder, excluding .ipynb_checkpoints.
    """

    # Check if path is absolute
    if not os.path.isabs(folder_path):
        # Adjust path based on current working directory
        folder_path = os.path.join(os.getcwd(), folder_path)

    # Local access, no authentication needed
    if not os.path.exists(folder_path):
        print(f"Folder not found: {folder_path}")
        return []

    folder_paths = [os.path.join(folder_path, directory)
                    for directory in os.listdir(folder_path)
                    if os.path.isdir(os.path.join(folder_path, directory))
                    and not directory.startswith(".ipynb_checkpoints")]

    return folder_paths


def get_tsv_file_paths(folder_path):
    """
    Gets all the paths of .tsv files within a specific folder, excluding .ipynb_checkpoints.

    Args:
        folder_path (str): The path to the folder.

    Returns:
        list: A list of all .tsv file paths within the specified folder, excluding .ipynb_checkpoints.
    """

    tsv_file_paths = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".tsv") and not filename.startswith(".ipynb_checkpoints"):
            file_path = os.path.join(folder_path, filename)
            tsv_file_paths.append(file_path)

    return tsv_file_paths



def find_unique_combinations(inputs, combination_length=2):
    """
    Function to find unique pairs of the inputs provided.
    Args:
        inputs(list): The list of inputs to find the unique pairs of
         combination_length: the length of the unique combinations
         (default=2)
    
    """

    unique_combinations = set(combinations(inputs, combination_length))
    return unique_combinations

def get_file_name(file_path):
  
    """
    Extracts the filename from a given file path.

    Args:
        file_path (str): The path to the file.

    Returns:
        str: The filename without the directory path.
    """
    file_name=os.path.basename(file_path)
    # components=file_name.split("_")
    # sample_name= components[0]+"_"+components[1]

    return file_name




In [3]:
def merge_fil_in_and_out(file1,file2):
    """ 
    Function for merging the filtered in and filterd out files of a sample into a single dataframe.
    Args: 
    file1 and file2 : filtered in and filtered out files of the sample.
    Returns:
    merged_df: merged datframe containing filtered in and filtered out variant data
    """
    
    df1= pd.read_table(file1, sep='\t', skiprows=17, low_memory=False)
    df2= pd.read_table(file2, sep='\t', skiprows=17, low_memory=False)
    merged_df = pd.concat([df1, df2], axis=0, join='outer')  # keeping all columns
    return merged_df




In [4]:

def format_df_snv(sample_1,sample_2):
    """
    Removes variants with filter marked as "NOCALL" and "FAIL"
    Adds "Chr_pos_ref_alt" column which acts as match point
    Adds "Match_with_comparing_run" and populates it with True if the variants are found in both files
    Merges the dataframe based on Chr_pos_ref_alt
    
    Args:
        sample_1 and sample_2 which are the merged dataframes of the filtered in and filtered out variant data of the samples to be compared
    Returns:
        Each of the modified sample dataframes and the merged dataframe
    """
    
    filtered_df_sample1 = sample_1[(sample_1["Filter"] != "NOCALL") & (sample_1["Filter"] != "FAIL")]
    filtered_df_sample1.insert(5, 'Chr_pos_ref_alt', filtered_df_sample1['Locus'].fillna('') + ":"+ filtered_df_sample1['Ref'].fillna('') + ":" + filtered_df_sample1['Observed Allele'].fillna(''))
    filtered_df_sample1.insert(6, "Match_with_comparing_run", value=None)
    
    
    filtered_df_sample2 = sample_2[(sample_2["Filter"] != "NOCALL") & (sample_2["Filter"] != "FAIL")]
    filtered_df_sample2.insert(5, 'Chr_pos_ref_alt', filtered_df_sample2['Locus'].fillna('') + ":"+ filtered_df_sample2['Ref'].fillna('') + ":" + filtered_df_sample2['Observed Allele'].fillna(''))
    filtered_df_sample2.insert(6, "Match_with_comparing_run", value=None)
    
    
    
    filtered_df_sample1.loc[:, "Match_with_comparing_run"] = filtered_df_sample1.loc[:,"Chr_pos_ref_alt"].isin(filtered_df_sample2.loc[:,"Chr_pos_ref_alt"])
    filtered_df_sample2.loc[:, "Match_with_comparing_run"] = filtered_df_sample2.loc[:,"Chr_pos_ref_alt"].isin(filtered_df_sample1.loc[:,"Chr_pos_ref_alt"])
    
    merged_df_type = pd.merge(filtered_df_sample1, filtered_df_sample2, on="Chr_pos_ref_alt")
    print("Sample 1 and Sample 2 dataframes formatted")
    return filtered_df_sample1,filtered_df_sample2,merged_df_type

def cal_percent_match_snv(filtered_df_sample1,filtered_df_sample2,merged_df_type):
    """
    Calculates the percentage match and returns the required data for summary sheet.
    Args:
        filtered_df_sample1 : Modified dataframe of sample 1
        filtered_df_sample2 : Modified dataframe of sample 2
        merged_df_type : Merged dataframe of sample 1 and 2 based on Chr_pos_ref_alt
    Returns:
        no_of_snvs1 (int)= The total number of SNVs in sample 1
        no_of_snvs2 (int)= The total number of SNVs in sample 2
        no_of_snv_matches (int) = The total number of SNVs that are present in both sample 1 and 2 (matches)
        percent_match (int) = The percentage match of SNVs between both the samples
    """

    no_of_snvs1= filtered_df_sample1['Type'].value_counts()['SNV']
    no_of_snvs2= filtered_df_sample2['Type'].value_counts()['SNV']
    
    no_of_snv_matches = ((merged_df_type["Match_with_comparing_run_x"] == True) &
                       (merged_df_type['Type_x'] == "SNV") &  # Check SNV in DataFrame 1
                       (merged_df_type['Type_y'] == "SNV")).sum()  # Check SNV in DataFrame 2
    
    
    percent_match= round((no_of_snv_matches*2*100/(no_of_snvs1+no_of_snvs2)),2)
    print("Required stats successfully calculated")
    return no_of_snvs1,no_of_snvs2,no_of_snv_matches,percent_match

def create_af_comp_df_snv(filtered_df_sample1, filtered_df_sample2,merged_df_type,sample1_name,sample2_name):
    """
    Creates a combined dataframe with the variant matches and unique variants in each sample to aid to R square calculations.
    For variants unique to a specific sample the corresponding allelic frequency of that sample is taken and the allelic frequency of the variant 
    is marked as 0 for the sample in which it is not present.
    Args:
        filtered_df_sample1 : Modified dataframe of sample 1
        filtered_df_sample2 : Modified dataframe of sample 2
        merged_df_type : Merged dataframe of sample 1 and 2 based on Chr_pos_ref_alt
        sample1_name (str)  = Name of sample 1 (folder name)
        sample2_name (str) = Name of sample 2 (folder name)
    Returns:
        combined_af_df: Dataframe with the variants and corresponding allelic frequencies  
    """
    desired_columns = [
        "Chr_pos_ref_alt",
        "Allele Frequency %_x",
        "Allele Frequency %_y"
    
    ]
    
    
    af_comp_df = merged_df_type[
        (merged_df_type["Match_with_comparing_run_x"] == True) & (merged_df_type["Type_x"] == "SNV") & (merged_df_type["Type_y"] == "SNV")
    ][desired_columns]
    #print(af_comp_df)
    
    desired_columns1 = ["Chr_pos_ref_alt", "Allele Frequency %"]
    af_unique1 = filtered_df_sample1.loc[
        (filtered_df_sample1["Match_with_comparing_run"] != True) & (filtered_df_sample1["Type"] == "SNV"),
        desired_columns1
    ]
    af_unique1 = af_unique1.rename(columns={'Allele Frequency %': 'Allele Frequency %_x'})
    af_unique1["Allele Frequency %_y"]=0
    #print(af_unique1)
    
    
    af_unique2 = filtered_df_sample2.loc[
        (filtered_df_sample2["Match_with_comparing_run"] != True) & (filtered_df_sample2["Type"] == "SNV"),
        desired_columns1
    ]
    af_unique2 = af_unique2.rename(columns={'Allele Frequency %': 'Allele Frequency %_y'})
    af_unique2.insert(1, "Allele Frequency %_x", value=0)
    #print(af_unique2)
    combined_af_df = pd.concat([af_comp_df, af_unique1,af_unique2], axis=0)
    combined_af_df = combined_af_df.rename(columns={ 'Allele Frequency %_x':f'Allele Frequency % {sample1_name}'})
    combined_af_df = combined_af_df.rename(columns={ 'Allele Frequency %_y':f'Allele Frequency % {sample2_name}'})
    print("Dataframe required for plotting scatter plot formed")
    return combined_af_df
    







In [5]:

def format_df_snv_and_refsnv(sample_1,sample_2):
    """

    Removes variants with filter marked as "NOCALL" and "FAIL"
    Adds "Chr_pos_ref_alt" column which acts as match point
    Adds "Match_with_comparing_run" and populates it with True if the variants are found in both files
    Adds a "User_defined_type_column" which identifies the SNVs of type REF and the SNVs of type SNV.
    The "User_defined_type" column will hence have "SNV" marked for all variants whose type is SNV in the type column and SNVs whose type is REF.
    Merges the dataframe based on Chr_pos_ref_alt
    
    Args:
        sample_1 and sample_2 which are the merged dataframes of the filtered in and filtered out variant data of the samples to be compared
    Returns:
        Each of the modified sample dataframes and the merged dataframe type column and SNVs whose type is REF.
    """
    filtered_df_sample1 = sample_1[(sample_1["Filter"] != "NOCALL") & (sample_1["Filter"] != "FAIL")]
    filtered_df_sample1.insert(5, 'Chr_pos_ref_alt', filtered_df_sample1['Locus'].fillna('') + ":"+ filtered_df_sample1['Ref'].fillna('') + ":" + filtered_df_sample1['Observed Allele'].fillna(''))
    filtered_df_sample1.insert(6, "Match_with_comparing_run", value=None)
    
    
    filtered_df_sample2 = sample_2[(sample_2["Filter"] != "NOCALL") & (sample_2["Filter"] != "FAIL")]
    filtered_df_sample2.insert(5, 'Chr_pos_ref_alt', filtered_df_sample2['Locus'].fillna('') + ":"+ filtered_df_sample2['Ref'].fillna('') + ":" + filtered_df_sample2['Observed Allele'].fillna(''))
    filtered_df_sample2.insert(6, "Match_with_comparing_run", value=None)
    
    
    
    filtered_df_sample1.loc[:, "Match_with_comparing_run"] = filtered_df_sample1.loc[:,"Chr_pos_ref_alt"].isin(filtered_df_sample2.loc[:,"Chr_pos_ref_alt"])
    filtered_df_sample2.loc[:, "Match_with_comparing_run"] = filtered_df_sample2.loc[:,"Chr_pos_ref_alt"].isin(filtered_df_sample1.loc[:,"Chr_pos_ref_alt"])
    filtered_df_sample1.insert(6, "User_defined_type", value=None)
    filtered_df_sample2.insert(6, "User_defined_type", value=None)
    
    filtered_df_sample1.loc[filtered_df_sample1['Type'] == "SNV", "User_defined_type"] = "SNV"
    filtered_df_sample2.loc[filtered_df_sample2['Type'] == "SNV", "User_defined_type"] = "SNV"


    filtered_df_sample1.loc[(filtered_df_sample1['Type'] == 'REF') & (filtered_df_sample1['Observed Allele'].str.len() == 1) & (filtered_df_sample1['Ref'].str.len() == 1) & (filtered_df_sample1['Ref']!= filtered_df_sample1['Observed Allele']), "User_defined_type"] = "SNV"
    filtered_df_sample2.loc[(filtered_df_sample2['Type'] == 'REF') & (filtered_df_sample2['Observed Allele'].str.len() == 1) & (filtered_df_sample2['Ref'].str.len() == 1) & (filtered_df_sample2['Ref']!= filtered_df_sample2['Observed Allele']), "User_defined_type"] = "SNV"
    filtered_df_sample1["User_defined_type"].fillna("NA")
    filtered_df_sample2["User_defined_type"].fillna("NA")


    merged_df_type = pd.merge(filtered_df_sample1, filtered_df_sample2, on="Chr_pos_ref_alt")
    print("Sample 1 and Sample 2 dataframes formatted")
    return filtered_df_sample1,filtered_df_sample2,merged_df_type

def cal_percent_match_snv_and_refsnv(filtered_df_sample1,filtered_df_sample2,merged_df_type):
    """
    Calculates the percentage match and returns the required data for summary sheet.
    Args:
        filtered_df_sample1 : Modified dataframe of sample 1
        filtered_df_sample2 : Modified dataframe of sample 2
        merged_df_type : Merged dataframe of sample 1 and 2 based on Chr_pos_ref_alt
    Returns:
        no_of_snvs1 (int)= The total number of SNVs in sample 1
        no_of_snvs2 (int)= The total number of SNVs in sample 2
        no_of_snv_matches (int) = The total number of SNVs that are present in both sample 1 and 2 (matches)
        percent_match (int) = The percentage match of SNVs between both the samples
    """

    no_of_snvs1= filtered_df_sample1['User_defined_type'].value_counts()['SNV']
    no_of_snvs2= filtered_df_sample2['User_defined_type'].value_counts()['SNV']
    
    no_of_snv_matches = ((merged_df_type["Match_with_comparing_run_x"] == True) &
                       (merged_df_type['User_defined_type_x'] == "SNV") &  # Check SNV in DataFrame 1
                       (merged_df_type['User_defined_type_y'] == "SNV")).sum()  # Check SNV in DataFrame 2
    
    
    percent_match= round((no_of_snv_matches*2*100/(no_of_snvs1+no_of_snvs2)),2)
    print("Required stats successfully calculated")
    return no_of_snvs1,no_of_snvs2,no_of_snv_matches,percent_match

def create_af_comp_df_snv_and_refsnv(filtered_df_sample1, filtered_df_sample2,merged_df_type,sample1_name,sample2_name):
    """
    Creates a combined dataframe with the variant matches and unique variants in each sample to aid to R square calculations.
    For variants unique to a specific sample the corresponding allelic frequency of that sample is taken and the allelic frequency of the variant 
    is marked as 0 for the sample in which it is not present.
    Args:
        filtered_df_sample1 : Modified dataframe of sample 1
        filtered_df_sample2 : Modified dataframe of sample 2
        merged_df_type : Merged dataframe of sample 1 and 2 based on Chr_pos_ref_alt
        sample1_name (str)  = Name of sample 1 (folder name)
        sample2_name (str) = Name of sample 2 (folder name)
    Returns:
        combined_af_df: Dataframe with the variants and corresponding allelic frequencies  
    """
    desired_columns = [
        "Chr_pos_ref_alt",
        "Allele Frequency %_x",
        "Allele Frequency %_y"
    
    ]
    
    
    af_comp_df = merged_df_type[
        (merged_df_type["Match_with_comparing_run_x"] == True) & (merged_df_type["User_defined_type_x"] == "SNV") & (merged_df_type["User_defined_type_y"] == "SNV")
    ][desired_columns]
    #print(af_comp_df)
    
    desired_columns1 = ["Chr_pos_ref_alt", "Allele Frequency %"]
    af_unique1 = filtered_df_sample1.loc[
        (filtered_df_sample1["Match_with_comparing_run"] != True) & (filtered_df_sample1["User_defined_type"] == "SNV"),
        desired_columns1
    ]
    af_unique1 = af_unique1.rename(columns={'Allele Frequency %': 'Allele Frequency %_x'})
    af_unique1["Allele Frequency %_y"]=0
    #print(af_unique1)
    
    
    af_unique2 = filtered_df_sample2.loc[
        (filtered_df_sample2["Match_with_comparing_run"] != True) & (filtered_df_sample2["User_defined_type"] == "SNV"),
        desired_columns1
    ]
    af_unique2 = af_unique2.rename(columns={'Allele Frequency %': 'Allele Frequency %_y'})
    af_unique2.insert(1, "Allele Frequency %_x", value=0)
    #print(af_unique2)
    combined_af_df = pd.concat([af_comp_df, af_unique1,af_unique2], axis=0)
    combined_af_df = combined_af_df.rename(columns={ 'Allele Frequency %_x':f'Allele Frequency % {sample1_name}'})
    combined_af_df = combined_af_df.rename(columns={ 'Allele Frequency %_y':f'Allele Frequency % {sample2_name}'})
    print("Dataframe required for plotting scatter plot formed")
    return combined_af_df

In [6]:

def scatter_plot(combined_af_df,sample1_name,sample2_name):
    """
    Calculates R square value of the corresponding allelic frequency , plots a scatter plot and saves it as an image.
    Args:
        combined_af_df: Dataframe with the variants and corresponding allelic frequencies 
        sample1_name (str)  = Name of sample 1 (folder name)
        sample2_name (str) = Name of sample 2 (folder name)
    Returns: 
        r_squared = R square value
        img = The scatter plot saved as an image
    """
    x_data= combined_af_df[f'Allele Frequency % {sample1_name}']
    y_data= combined_af_df[f'Allele Frequency % {sample2_name}']
    correlation_matrix = np.corrcoef(x_data, y_data)
    r_squared = correlation_matrix[0, 1] ** 2
    
    plt.figure(figsize=(6, 4))  # Set custom figure size
    plt.scatter(x_data, y_data,color='#F4C2C2')
    plt.xlabel(f"Allelic frequency from sample 1 {sample1_name}", fontsize=8)
    plt.ylabel(f"Allelic frequency from sample 2 {sample2_name}", fontsize=8)
    plt.title(f"Allelic Frequency Comparison {sample1_name} vs {sample2_name} ", fontsize=8)
    plt.annotate(f"R² = {r_squared:.4f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=10)
    
    filename = f"{results_folder_path}/scatter_plot_{sample1_name}_{sample2_name}.jpg"

    plt.savefig(filename, dpi=300)
    img = filename
    plt.close()
    print("Scatter plot plotted succesfully")
    return r_squared,img
    

In [7]:
def cal_snv():
    """
    Sequence of steps for method 1
    """
# Create a new workbook and worksheet
    workbook = xlsxwriter.Workbook(f"{results_folder_path}/Results_snv_{get_file_name(results_folder_path)}.xlsx")
    worksheet3 = workbook.add_worksheet("Result_summary")
    sample1_name_list=[]
    sample2_name_list=[]
    r_squared_list=[]
    no_snvs1=[]
    no_snvs2=[]
    no_of_matches=[]
    percent_match=[]
    result_summary_df=pd.DataFrame()
    for combos in unique_combinations:
        fil_in_and_out1=get_tsv_file_paths(combos[0])
        fil_in_and_out2=get_tsv_file_paths(combos[1])
        sample1_name= get_file_name(combos[0])
        sample1_name_list.append(sample1_name)
        sample2_name=get_file_name(combos[1])
        sample2_name_list.append(sample2_name)
        sample_1=merge_fil_in_and_out(fil_in_and_out1[0],fil_in_and_out1[1])
        sample_2=merge_fil_in_and_out(fil_in_and_out2[0],fil_in_and_out2[1])
        print(".tsv files from both samples fetched. Filtered in and filtered out files successfully merged")
        work_df=format_df_snv(sample_1,sample_2)
        stats=cal_percent_match_snv(work_df[0],work_df[1],work_df[2])
        no_snvs1.append(stats[0])
        no_snvs2.append(stats[1])
        no_of_matches.append(stats[2])
        percent_match.append(stats[3])
        af_comp_df=create_af_comp_df_snv(work_df[0],work_df[1],work_df[2],sample1_name,sample2_name)
        worksheet = workbook.add_worksheet(f" {sample1_name} vs {sample2_name}")
        r_squared,img=scatter_plot(af_comp_df,sample1_name,sample2_name)
        image_list.append(img)
        r_squared_list.append(r_squared)
        worksheet.insert_image('E1',img)
        row = 0
        col = 0
        for col_name in af_comp_df.columns:
            worksheet.write(row, col, col_name)
            col += 1
        # Write DataFrame data
        row += 1  # Move to the next row after headers
        for index, row_data in af_comp_df.iterrows():
            col = 0
            for value in row_data:
                worksheet.write(row, col, value)
                col += 1
            row+=1
        #image_path = os.path.abspath(img.filename)
        
        
        
        
    result_summary_df["Sample 1"]= sample1_name_list
    result_summary_df["Sample 2"]=sample2_name_list
    result_summary_df["Number of SNVs in sample 1"]=no_snvs1
    result_summary_df["Number of SNVs in sample 2"]=no_snvs2
    result_summary_df["Number of SNV match between sample 1 and 2"]=no_of_matches
    result_summary_df["Percent SNV match"]=percent_match
    result_summary_df["R_square_value"]=r_squared_list
    #print(result_summary_df)
    
    
    row = 0
    col = 0
    for col_name in result_summary_df.columns:
        worksheet3.write(row, col, col_name)
        col += 1
    
    # Write DataFrame data
    row += 1  # Move to the next row after headers
    for index, row_data in result_summary_df.iterrows():
        col = 0
        for value in row_data:
            worksheet3.write(row, col, value)
            col += 1
        row+=1
    
    # Close the workbook
    workbook.close()
    # for image in image_names:
    #     os.remove(image)
    
    

    

In [8]:
def cal_snv_and_refsnv():
    """
    Sequence of stpes for method 2
    """
# Create a new workbook and worksheet
    workbook1 = xlsxwriter.Workbook(f"{results_folder_path}/Results_snv_and_refsnv_{get_file_name(results_folder_path)}.xlsx")
    worksheet1 = workbook1.add_worksheet("Result_summary")
    sample1_name_list=[]
    sample2_name_list=[]
    r_squared_list=[]
    no_snvs1=[]
    no_snvs2=[]
    no_of_matches=[]
    percent_match=[]
    result_summary_df1=pd.DataFrame()
    #print(unique_combinations)
    for combos in unique_combinations:
        fil_in_and_out1=get_tsv_file_paths(combos[0])
        fil_in_and_out2=get_tsv_file_paths(combos[1])
        sample1_name= get_file_name(combos[0])
        sample1_name_list.append(sample1_name)
        sample2_name=get_file_name(combos[1])
        sample2_name_list.append(sample2_name)
        sample_1=merge_fil_in_and_out(fil_in_and_out1[0],fil_in_and_out1[1])
        sample_2=merge_fil_in_and_out(fil_in_and_out2[0],fil_in_and_out2[1])
        work_df=format_df_snv_and_refsnv(sample_1,sample_2)
        stats=cal_percent_match_snv_and_refsnv(work_df[0],work_df[1],work_df[2])
        no_snvs1.append(stats[0])
        no_snvs2.append(stats[1])
        no_of_matches.append(stats[2])
        percent_match.append(stats[3])
        af_comp_df=create_af_comp_df_snv_and_refsnv(work_df[0],work_df[1],work_df[2],sample1_name,sample2_name)
        r_squared,img=scatter_plot(af_comp_df,sample1_name,sample2_name)
        r_squared_list.append(r_squared)
        image_list.append(img)
        worksheet2 = workbook1.add_worksheet(f" {sample1_name} vs {sample2_name}")
        worksheet2.insert_image('E1',img)
        row = 0
        col = 0
        for col_name in af_comp_df.columns:
            worksheet2.write(row, col, col_name)
            col += 1
        # Write DataFrame data
        row += 1  # Move to the next row after headers
        for index, row_data in af_comp_df.iterrows():
            col = 0
            for value in row_data:
                worksheet2.write(row, col, value)
                col += 1
            row+=1
        #image_path = os.path.abspath(img.filename)
        
        
        
        
    result_summary_df1["Sample 1"]= sample1_name_list
    result_summary_df1["Sample 2"]=sample2_name_list
    result_summary_df1["Number of SNVs in sample 1"]=no_snvs1
    result_summary_df1["Number of SNVs in sample 2"]=no_snvs2
    result_summary_df1["Number of SNV match between sample 1 and 2"]=no_of_matches
    result_summary_df1["Percent SNV match"]=percent_match
    result_summary_df1["R_square_value"]=r_squared_list
    #print(result_summary_df1)
    
    
    row = 0
    col = 0
    for col_name in result_summary_df1.columns:
        worksheet1.write(row, col, col_name)
        col += 1
    
    # Write DataFrame data
    row += 1  # Move to the next row after headers
    for index, row_data in result_summary_df1.iterrows():
        col = 0
        for value in row_data:
            worksheet1.write(row, col, value)
            col += 1
        row+=1
    
    # Close the workbook
    workbook1.close()

    

In [9]:
import os


def check_folders_and_files():
    """Checks the input folder, if it has subfolders with 2 .tsv files each, handling exceptions and providing user options.

    Returns:
        str: The name of the main folder if successful, or "EXIT" if user chooses to exit.

    Raises:
        FileNotFoundError: If the main folder is not found.
        ValueError: If the main folder or any subfolder is empty,
            or if a subfolder doesn't contain exactly two TSV files.
    """

    while True:
        main_folder_name = input("Enter the name of the main folder with the samples: ")

        try:
            # Check for folder existence
            if not os.path.exists(main_folder_name):
                raise FileNotFoundError(f"Main folder '{main_folder_name}' not found.")

            # Check for empty folder
            if not os.listdir(main_folder_name):
                raise ValueError(f"Main folder '{main_folder_name}' is empty.")

            # Iterate through subfolders
            #print(os.listdir(main_folder_name)) 
            # length=0
            for subfolder_name in os.listdir(main_folder_name):
            #     if not subfolder_name.endswith(".ipynb_checkpoints"):
            #         length+=1
            #     if length ==1:
            #          raise ValueError(f"Not enough samples for comparison.")
                subfolder_path = os.path.join(main_folder_name, subfolder_name)

                # Exclude files and folders with .pynb_checkpoint extension
                if  subfolder_name.endswith(".pynb_checkpoints"):
                    continue

                # Check for subfolder existence, directory type, and .ipynb exclusion
                if not os.path.isdir(subfolder_path) or subfolder_name.endswith(".ipynb_checkpoints"):
                    continue
                

                # Check for empty subfolder
                if not os.listdir(subfolder_path):
                    raise ValueError(f"Subfolder '{subfolder_name}' is empty.")
                
                # Filter TSV files excluding .pynb_checkpoint
                tsv_files = [
                    f
                    for f in os.listdir(subfolder_path)
                    if f.endswith(".tsv") and not f.endswith(".pynb_checkpoint")
                ]

                # Check for exactly two TSV files
                if len(tsv_files) != 2:
                    raise ValueError(
                        f"Subfolder '{subfolder_name}' does not contain exactly two TSV files."
                    )

            # All checks passed, print success message and return folder name
            print("All expected folders and files found!")
            return main_folder_name

        except (FileNotFoundError, ValueError) as e:
            print(f"Error: {e}")

            # # Offer retry or exit options
            # choice = input("Do you want to retry (r) or exit (e)? ").lower()
            # if choice not in ("r", "e"):
            #     print("Invalid choice. Please enter 'r' or 'e'.")
            #     continue  # Repeat prompt if invalid choice

            # if choice == "e":
            #     return "EXIT"  # Indicate user chose to exit
            while True:
                choice = input("Do you want to retry (r) or exit (e)? ").lower()
                if choice not in ("r", "e"):
                    print("Invalid choice. Please enter 'r' or 'e'.")
                    continue  # Repeat prompt if invalid choice
    
                if choice == "e":
                    return "EXIT"  # Indicate user chose to exit
                else:
            # Retry: break out of the current iteration and re-prompt for folder name
                    break


In [10]:

def create_folder(folder_name):
  """Creates a unique folder in the same directory as the script with the given name, appending a counter if necessary.

  Args:
      folder_name: The base name of the folder to create.

  Returns:
      str: The full path to the created folder.
  """

  current_dir = os.getcwd()
  counter = 1

  while True:
    folder_path = os.path.join(current_dir, folder_name)
    if counter > 1:
      folder_path += f" ({counter})"

    try:
      os.mkdir(folder_path)
      print(f"Folder '{get_file_name(folder_path)}' created to store the results.")
      return folder_path
    except OSError as e:
      if e.errno == 17:  # File exists error
        counter += 1
      else:
        raise  # Re-raise other errors



In [11]:

image_list=[]
def get_user_choice():
    """
    Presents options, prompts user for input, validates input, and returns a valid choice.

    Returns:
        int or None: The valid user choice (1, 2, 3, ...) or None to exit.
    """

    valid_choices = [1, 2, 3]  # Replace with your valid choices

    while True:
        try:
            print("""
Choose the method to calculate % match and allele frequency concordance:
        1. Only SNVs
        2. All SNVs including SNVs of type REF
        3. Both 1 and 2
        4. Exit
                """)

            user_input = int(input("Enter a choice or '4' to exit: "))
            if user_input == 4:
                return None  # Exit option
            elif user_input in valid_choices:
                return user_input
            else:
                print("Invalid choice. Please enter one of the following: ",
                      ", ".join(map(str, valid_choices)))
        except ValueError:
            print("Invalid input. Please enter a number.")
            
""" While loop asking for user input for the folder name with the samples and method"""
while True:
    folder_path = check_folders_and_files()
    if folder_path !="EXIT":
        
        # Call the function to get the user's choice
        user_choice = get_user_choice()
        
        
        # Use the user_choice in your subsequent code logic
        if user_choice == 1:
            results_folder_path=create_folder(f"{folder_path}_results")
        #get all the folders paths in the entered folder to find unique combinations
            folder_paths=get_folder_paths(folder_path)
            unique_combinations=find_unique_combinations(folder_paths)
            print("Fetching results..")
            cal_snv()
            print(f"Results have been saved to Results_snv_{get_file_name(folder_path)}.xlsx at {get_file_name(results_folder_path)}")
            
        
        elif user_choice == 2:
            results_folder_path=create_folder(f"{folder_path}_results")
        #get all the folders paths in the entered folder to find unique combinations
            folder_paths=get_folder_paths(folder_path)
            unique_combinations=find_unique_combinations(folder_paths)
            print("Fetching results..")
            cal_snv_and_refsnv()
            print(f"Results have been saved to Results_snv_and_refsnv_{get_file_name(folder_path)}.xlsx at {get_file_name(results_folder_path)}")
            
        elif user_choice == 3:
            results_folder_path=create_folder(f"{folder_path}_results")
        #get all the folders paths in the entered folder to find unique combinations
            folder_paths=get_folder_paths(folder_path)
            unique_combinations=find_unique_combinations(folder_paths)
            print("Fetching results..")
            cal_snv()
            cal_snv_and_refsnv()
            print(f"Results have been saved to Results_snv_{get_file_name(folder_path)}.xlsx and Results_snv_and_refsnv_{get_file_name(folder_path)}.xlsx at {get_file_name(results_folder_path)}")
        elif user_choice == None:
            print("Exiting...")
            break
        continue_choice = input("Do you want to continue working on another set of samples? (y/n): ")
        if continue_choice.lower() != "y":
            print("Exited")
            break
    else:
        print ("Exited")
        break
        
""" for loop for deleting intermediate scatter plot images generated while running""" 
#print (image_list)
for image in image_list:
    if os.path.exists(image):
        os.remove(image)

    
        

Enter the name of the main folder with the samples:  test


All expected folders and files found!

Choose the method to calculate % match and allele frequency concordance:
        1. Only SNVs
        2. All SNVs including SNVs of type REF
        3. Both 1 and 2
        4. Exit
                


Enter a choice or '4' to exit:  3


Folder 'test_results' created to store the results.
Fetching results..
.tsv files from both samples fetched. Filtered in and filtered out files successfully merged
Sample 1 and Sample 2 dataframes formatted
Required stats successfully calculated
Dataframe required for plotting scatter plot formed
Scatter plot plotted succesfully
.tsv files from both samples fetched. Filtered in and filtered out files successfully merged
Sample 1 and Sample 2 dataframes formatted
Required stats successfully calculated
Dataframe required for plotting scatter plot formed
Scatter plot plotted succesfully
.tsv files from both samples fetched. Filtered in and filtered out files successfully merged
Sample 1 and Sample 2 dataframes formatted
Required stats successfully calculated
Dataframe required for plotting scatter plot formed
Scatter plot plotted succesfully
Sample 1 and Sample 2 dataframes formatted
Required stats successfully calculated
Dataframe required for plotting scatter plot formed
Scatter plot pl

Do you want to continue working on another set of samples? (y/n):  n


Exited
